# OPERA — results

Straight from `results/*.csv` (regenerate with `python scripts/summarize_results.py`).

* **Forecasting:** MSE, MAE — lower is better
* **Classification:** AUROC, F1 (macro) — higher is better. PAM is 8-class, so it has no AUROC
* **Shifted** = average over the 4 shift conditions (density, timing, variables, composed)
* **TMix is excluded** from these results (it failed to train reliably). Its raw runs remain in `results/raw/`;
  `python scripts/summarize_results.py --include-tmix` puts it back

For every metric, clean and shifted each get three tables:

1. **Values** — mean ± std over seeds, with an **AVERAGE** row
2. **% improvement vs baseline** — paired on matching seeds, **positive = better** (lower MSE/MAE, higher AUROC/F1), with an **AVERAGE** row
3. **Summary per method** — average value, average % improvement, and **win counts** against baseline:
   *seed wins* = individual runs that beat the baseline run with the same setting, backbone and seed;
   *cell wins* = setting × backbone cells whose mean beats the baseline mean

**Note on seed counts:** OPERA and baseline currently have seeds 0–4, while OPERA++ and mining have seeds
1–4 (seed 0 is being retrained), so their seed-win denominators differ (e.g. 60 vs 48).

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results").is_dir())
R = ROOT / "results" / "cu_ce9a1cc"
fc = pd.concat([pd.read_csv(R / "forecast_clean.csv"), pd.read_csv(R / "forecast_shift.csv")])
cl = pd.concat([pd.read_csv(R / "classify_clean.csv"), pd.read_csv(R / "classify_shift.csv")])
seed = pd.read_csv(R / "per_seed_results.csv")

# --- what to show -------------------------------------------------------------------------------
GROUP     = "main"   # main | horizon | native_shift | irregular_backbone
BACKBONES = None     # None = all, e.g. ["itransformer"] for one backbone
METHODS   = ["vanilla", "opera", "opera++", "abl_opera++_scalar", "abl_mining"]
# -------------------------------------------------------------------------------------------------
LOWER_BETTER = {"mse", "mae"}
SHIFTS = ["density", "timing", "variables", "composed"]
pd.set_option("display.max_rows", 300); pd.set_option("display.width", 250)

def groups(grp=None):  # OPERA++ runs use the main settings, so show them alongside "main"
    grp = grp or GROUP
    return [grp, "opera++_study"] if grp == "main" else [grp]

def subset(df, condition, grp=None):
    d = df[df.group.isin(groups(grp)) & (df.condition == condition) & df.method.isin(METHODS)]
    return d[d.backbone.isin(BACKBONES)] if BACKBONES else d

def order(t):
    return t[[m for m in METHODS if m in t.columns]]

def seed_values(metric, condition, grp=None):
    """Per-run value: the clean metric, or the mean over the 4 shift conditions."""
    s = seed[seed.group.isin(groups(grp)) & seed.method.isin(METHODS)]
    if BACKBONES: s = s[s.backbone.isin(BACKBONES)]
    conds = ["clean"] if condition == "clean" else SHIFTS
    s = s[s.condition.isin(conds)].dropna(subset=[metric])
    v = s.groupby(["setting", "backbone", "method", "seed"])[metric].agg(["mean", "size"])
    return v[v["size"] == len(conds)]["mean"].unstack("method")   # rows: setting, backbone, seed

def show(df, metric, digits, grp=None):
    sign = -1 if metric in LOWER_BETTER else 1
    for cond, name in (("clean", "Clean (no shift)"), ("mean_4shifts", "Shifted (mean of 4 shifts)")):
        d = subset(df, cond, grp).dropna(subset=[f"{metric}_mean"])
        if d.empty: continue
        idx = ["setting", "backbone"]
        mean = order(d.pivot_table(index=idx, columns="method", values=f"{metric}_mean"))
        std = order(d.pivot_table(index=idx, columns="method", values=f"{metric}_std"))
        imp = order(d.pivot_table(index=idx, columns="method", values=f"{metric}_vs_baseline_pct"))

        f = f"{{:.{digits}f}}"
        vals = mean.map(f.format) + " ± " + std.map(f.format)
        vals = vals.where(mean.notna(), "")
        vals.loc[("AVERAGE", ""), :] = mean.mean().map(f.format)
        print(f"\n========== {metric.upper()} — {name} ==========")
        print("Values (mean ± std over seeds)"); display(vals)

        imp = imp.drop(columns="vanilla", errors="ignore")
        imp.loc[("AVERAGE", ""), :] = imp.mean()
        print("% improvement vs baseline (positive = better)"); display(imp.round(2))

        sv = seed_values(metric, "clean" if cond == "clean" else "shift", grp)
        rows = {}
        for m in mean.columns:
            r = {"avg value": f.format(mean[m].mean())}
            if m != "vanilla" and "vanilla" in sv:
                p = sv[[m, "vanilla"]].dropna()
                sw = int((sign * (p[m] - p["vanilla"]) > 0).sum())
                both = mean[[m, "vanilla"]].dropna()
                cw = int((sign * (both[m] - both["vanilla"]) > 0).sum())
                r.update({"avg % improvement": f"{imp[m].drop(('AVERAGE', ''), errors='ignore').mean():+.2f}%",
                          "seed wins": f"{sw}/{len(p)}", "seed win %": f"{100 * sw / max(len(p), 1):.0f}%",
                          "cell wins": f"{cw}/{len(both)}"})
            rows[m] = r
        print("Summary per method"); display(pd.DataFrame(rows).T.fillna(""))

## Forecasting — MSE

In [2]:
show(fc, "mse", 5)


========== MSE — Clean (no shift) ==========
Values (mean ± std over seeds)


method                                  vanilla              opera            opera++ abl_opera++_scalar         abl_mining
setting         backbone                                                                                                   
activity_1000ms itransformer  0.00266 ± 0.00005  0.00287 ± 0.00004  0.00286 ± 0.00007  0.00280 ± 0.00004  0.00295 ± 0.00004
                patchtst      0.00287 ± 0.00004  0.00307 ± 0.00009  0.00300 ± 0.00004  0.00300 ± 0.00002  0.00303 ± 0.00010
physionet_24h   itransformer  0.00547 ± 0.00014  0.00550 ± 0.00014  0.00550 ± 0.00005  0.00557 ± 0.00005  0.00558 ± 0.00018
                patchtst      0.01157 ± 0.00008  0.01204 ± 0.00029  0.01178 ± 0.00008  0.01184 ± 0.00009  0.01372 ± 0.00050
ushcn_12mo      itransformer  0.52247 ± 0.00151  0.51597 ± 0.00385  0.51624 ± 0.00159  0.51526 ± 0.00126  0.51858 ± 0.00342
                patchtst      0.51645 ± 0.00371  0.52334 ± 0.00305  0.52585 ± 0.00561  0.52453 ± 0.00267  0.52670 ± 0.00551
ushcn_1mo       itransformer  0.52948 ± 0.01180  0.51638 ± 0.00428  0.51923 ± 0.00534  0.51428 ± 0.00502  0.51903 ± 0.00389
                patchtst      0.51211 ± 0.00182  0.51580 ± 0.00452  0.50971 ± 0.00311  0.51291 ± 0.00182  0.51042 ± 0.00218
AVERAGE                                 0.26288            0.26187            0.26177            0.26127            0.26250

% improvement vs baseline (positive = better)


method                        opera  opera++  abl_opera++_scalar  abl_mining
setting         backbone                                                    
activity_1000ms itransformer  -7.31    -7.24               -5.29      -10.52
                patchtst      -7.00    -4.35               -4.25       -4.39
physionet_24h   itransformer  -2.46    -0.95               -1.65       -1.64
                patchtst      -2.92    -2.05               -2.64      -18.26
ushcn_12mo      itransformer   1.24     1.37                1.26        0.75
                patchtst      -1.43    -1.80               -1.44       -1.97
ushcn_1mo       itransformer   1.86     1.66                2.46        1.01
                patchtst      -0.71     0.65               -0.34        0.32
AVERAGE                       -2.34    -1.59               -1.49       -4.34

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.26288,,,,
opera,0.26187,-2.34%,12/40,30%,2/8
opera++,0.26177,-1.59%,12/32,38%,3/8
abl_opera++_scalar,0.26127,-1.49%,9/32,28%,2/8
abl_mining,0.26250,-4.34%,13/32,41%,3/8


... (2 outputs left out)

% improvement vs baseline (positive = better)


method                        opera  opera++  abl_opera++_scalar  abl_mining
setting         backbone                                                    
activity_1000ms itransformer  93.46    92.99               93.20       94.02
                patchtst      91.84    90.97               89.89       93.09
physionet_24h   itransformer  63.81    64.04               63.96       66.28
                patchtst      30.43    35.82               30.12       34.82
ushcn_12mo      itransformer   3.55     3.40                2.97        4.28
                patchtst       3.57     2.35                2.44        4.99
ushcn_1mo       itransformer   3.63     4.06                4.69        4.50
                patchtst       4.06     5.43                4.71        6.85
AVERAGE                       36.79    37.38               36.50       38.60

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.32179,,,,
opera,0.28950,+36.79%,40/40,100%,8/8
opera++,0.28755,+37.38%,32/32,100%,8/8
abl_opera++_scalar,0.28904,+36.50%,32/32,100%,8/8
abl_mining,0.28380,+38.60%,32/32,100%,8/8


## Forecasting — MAE

In [3]:
show(fc, "mae", 5)


========== MAE — Clean (no shift) ==========
Values (mean ± std over seeds)


method                                  vanilla              opera            opera++ abl_opera++_scalar         abl_mining
setting         backbone                                                                                                   
activity_1000ms itransformer  0.03202 ± 0.00054  0.03341 ± 0.00024  0.03323 ± 0.00062  0.03285 ± 0.00031  0.03425 ± 0.00029
                patchtst      0.03260 ± 0.00055  0.03385 ± 0.00124  0.03303 ± 0.00049  0.03295 ± 0.00045  0.03317 ± 0.00111
physionet_24h   itransformer  0.03805 ± 0.00057  0.03799 ± 0.00045  0.03833 ± 0.00044  0.03934 ± 0.00092  0.03897 ± 0.00108
                patchtst      0.05451 ± 0.00135  0.05752 ± 0.00267  0.05658 ± 0.00174  0.05586 ± 0.00153  0.06861 ± 0.00342
ushcn_12mo      itransformer  0.32375 ± 0.00810  0.31759 ± 0.00490  0.31592 ± 0.00645  0.31723 ± 0.00609  0.32164 ± 0.00555
                patchtst      0.30461 ± 0.01047  0.30674 ± 0.00416  0.31616 ± 0.00973  0.31172 ± 0.00734  0.31788 ± 0.00928
ushcn_1mo       itransformer  0.32423 ± 0.00731  0.31420 ± 0.00744  0.31702 ± 0.00460  0.31647 ± 0.00671  0.32229 ± 0.00590
                patchtst      0.30860 ± 0.01082  0.30621 ± 0.00867  0.30833 ± 0.00633  0.30895 ± 0.00975  0.30866 ± 0.00563
AVERAGE                                 0.17730            0.17594            0.17732            0.17692            0.18068

% improvement vs baseline (positive = better)


method                        opera  opera++  abl_opera++_scalar  abl_mining
setting         backbone                                                    
activity_1000ms itransformer  -4.73    -3.40               -2.87       -7.11
                patchtst      -3.57    -0.13               -1.32       -2.07
physionet_24h   itransformer  -0.00    -0.57               -3.77       -1.82
                patchtst      -3.05    -4.18               -3.25      -25.43
ushcn_12mo      itransformer   0.62     0.98                0.07       -0.78
                patchtst      -0.08    -1.70               -2.48       -4.12
ushcn_1mo       itransformer   1.51     2.71                1.71        0.06
                patchtst       1.23     1.20               -0.15        1.74
AVERAGE                       -1.01    -0.64               -1.50       -4.94

Summary per method

,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.17730,,,,
opera,0.17594,-1.01%,17/40,42%,4/8
opera++,0.17732,-0.64%,13/32,41%,3/8
abl_opera++_scalar,0.17692,-1.50%,9/32,28%,2/8
abl_mining,0.18068,-4.94%,8/32,25%,2/8


... (2 outputs left out)

% improvement vs baseline (positive = better)


method                        opera  opera++  abl_opera++_scalar  abl_mining
setting         backbone                                                    
activity_1000ms itransformer  80.52    80.27               80.21       81.48
                patchtst      79.02    77.81               77.07       80.06
physionet_24h   itransformer  42.94    42.94               41.92       44.45
                patchtst      27.46    28.28               27.04       24.66
ushcn_12mo      itransformer   1.75     2.82                1.20        2.07
                patchtst       4.79     2.43                1.64        1.92
ushcn_1mo       itransformer   6.39     6.77                4.89        7.15
                patchtst       6.80     7.29                5.44        7.25
AVERAGE                       31.21    31.08               29.93       31.13

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.26767,,,,
opera,0.20187,+31.21%,38/40,95%,8/8
opera++,0.20281,+31.08%,29/32,91%,8/8
abl_opera++_scalar,0.20454,+29.93%,30/32,94%,8/8
abl_mining,0.20159,+31.13%,31/32,97%,8/8


## Classification — AUROC

In [4]:
show(cl, "auroc", 2)


========== AUROC — Clean (no shift) ==========


Values (mean ± std over seeds)


method                     vanilla         opera    abl_mining
setting backbone                                              
P12     itransformer  84.11 ± 1.64  84.72 ± 0.85  85.09 ± 0.98
        patchtst      85.03 ± 0.97  85.75 ± 0.72  85.18 ± 0.65
P19     itransformer  87.30 ± 1.46  89.02 ± 1.77  89.01 ± 2.01
        patchtst      82.48 ± 1.11  83.57 ± 1.21  83.10 ± 0.45
AVERAGE                      84.73         85.77         85.60

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer   0.07        0.41
        patchtst       0.95        0.33
P19     itransformer   1.31        1.92
        patchtst       0.37        0.28
AVERAGE                0.67        0.73

Summary per method

... (3 outputs left out)

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer   0.65        2.16
        patchtst       1.70        1.60
P19     itransformer   3.52        3.38
        patchtst       2.26        1.56
AVERAGE                2.03        2.18

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,81.77,,,,
opera,83.76,+2.03%,20/20,100%,4/4
abl_mining,83.90,+2.18%,16/16,100%,4/4


## Classification — F1 (macro)

In [5]:
show(cl, "f1", 2)


========== F1 — Clean (no shift) ==========
Values (mean ± std over seeds)


method                     vanilla         opera    abl_mining
setting backbone                                              
P12     itransformer  68.24 ± 1.42  67.68 ± 1.69  67.72 ± 0.25
        patchtst      68.71 ± 1.80  69.37 ± 0.61  67.80 ± 0.75
P19     itransformer  66.93 ± 1.97  65.38 ± 2.14  66.46 ± 1.44
        patchtst      65.82 ± 2.76  62.45 ± 6.06  64.84 ± 5.46
PAM     itransformer  89.75 ± 0.80  89.83 ± 0.48  87.50 ± 0.38
        patchtst      88.40 ± 0.85  89.00 ± 0.63  86.37 ± 0.95
AVERAGE                      74.64         73.95         73.45

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer  -0.62       -0.08
        patchtst       1.03       -0.92
P19     itransformer  -1.64       -0.46
        patchtst      -7.84       -0.98
PAM     itransformer   0.04       -3.01
        patchtst      -0.41       -1.64
AVERAGE               -1.57       -1.18

Summary per method

,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,74.64,,,,
opera,73.95,-1.57%,11/26,42%,3/6
abl_mining,73.45,-1.18%,6/24,25%,0/6


... (3 outputs left out)

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer   1.70        0.13
        patchtst       1.53        0.52
P19     itransformer   1.14       -3.08
        patchtst      -4.95        2.51
PAM     itransformer  12.76       16.15
        patchtst       5.18        0.39
AVERAGE                2.89        2.77

Summary per method

,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,67.80,,,,
opera,70.29,+2.89%,21/26,81%,5/6
abl_mining,70.06,+2.77%,16/24,67%,5/6


## Optional — other classification metrics
Change the metric name to `acc`, `auprc`, `precision` or `recall`.

In [6]:
show(cl, "acc", 2)


========== ACC — Clean (no shift) ==========
Values (mean ± std over seeds)


method                     vanilla         opera    abl_mining
setting backbone                                              
P12     itransformer  78.88 ± 3.33  79.93 ± 2.12  80.07 ± 0.40
        patchtst      82.07 ± 1.66  82.49 ± 0.80  80.25 ± 0.92
P19     itransformer  90.24 ± 1.78  88.86 ± 2.04  89.98 ± 1.55
        patchtst      93.64 ± 2.21  94.84 ± 0.60  94.96 ± 0.69
PAM     itransformer  88.54 ± 0.91  88.54 ± 0.40  86.00 ± 0.38
        patchtst      87.08 ± 0.95  87.75 ± 0.54  84.60 ± 1.11
AVERAGE                      86.74         87.07         85.98

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer   0.91       -0.16
        patchtst      -0.20       -1.88
P19     itransformer  -0.35       -0.64
        patchtst       0.14        1.22
PAM     itransformer  -0.21       -3.46
        patchtst      -0.43       -2.04
AVERAGE               -0.02       -1.16

Summary per method

,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,86.74,,,,
opera,87.07,-0.02%,15/30,50%,4/6
abl_mining,85.98,-1.16%,6/24,25%,2/6


... (2 outputs left out)

% improvement vs baseline (positive = better)


method                opera  abl_mining
setting backbone                       
P12     itransformer   2.62       -1.03
        patchtst       0.47       -1.75
P19     itransformer   0.77       -2.33
        patchtst       0.20        0.63
PAM     itransformer  12.65       16.48
        patchtst       4.84       -0.82
AVERAGE                3.59        1.86

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,80.70,,,,
opera,83.62,+3.59%,26/30,87%,6/6
abl_mining,82.81,+1.86%,11/24,46%,3/6


# Irregular time-series methods — GRU-D, mTAND, Warpformer

The same OPERA comparison on three published irregular-time-series architectures, re-implemented at this
benchmark's shared budget (absolute numbers are therefore not comparable to their papers; the baseline vs OPERA
contrast within each method is). Same 7 settings, seeds 0–4.

## Irregular methods — MSE

In [7]:
show(fc, "mse", 5, grp="irregular_backbone")


========== MSE — Clean (no shift) ==========
Values (mean ± std over seeds)


method                                vanilla              opera
setting         backbone                                        
activity_1000ms grud        0.00292 ± 0.00002  0.00301 ± 0.00002
                mtand       0.00307 ± 0.00008  0.00310 ± 0.00003
                warpformer  0.01037 ± 0.00108  0.00873 ± 0.00092
physionet_24h   grud        0.00576 ± 0.00012  0.00567 ± 0.00012
                mtand       0.00653 ± 0.00013  0.00653 ± 0.00015
                warpformer  0.00844 ± 0.00014  0.00828 ± 0.00010
ushcn_12mo      grud        0.62322 ± 0.01619  0.66091 ± 0.01850
                mtand       0.62400 ± 0.04126  0.63149 ± 0.04488
                warpformer  0.68341 ± 0.04664  0.69312 ± 0.03810
ushcn_1mo       grud        0.52140 ± 0.00297  0.51782 ± 0.00331
                mtand       0.70096 ± 0.00187  0.69998 ± 0.00240
                warpformer  0.55597 ± 0.01047  0.55166 ± 0.00665
AVERAGE                               0.31217            0.31586

% improvement vs baseline (positive = better)


method                      opera
setting         backbone         
activity_1000ms grud        -3.02
                mtand       -0.91
                warpformer  17.13
physionet_24h   grud         1.87
                mtand        0.76
                warpformer   1.80
ushcn_12mo      grud        -5.60
                mtand       -0.97
                warpformer  -0.83
ushcn_1mo       grud         0.66
                mtand       -0.00
                warpformer   1.48
AVERAGE                      1.03

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.31217,,,,
opera,0.31586,+1.03%,29/60,48%,6/12


... (2 outputs left out)

% improvement vs baseline (positive = better)


method                      opera
setting         backbone         
activity_1000ms grud        76.19
                mtand       66.43
                warpformer  34.10
physionet_24h   grud        15.91
                mtand       16.02
                warpformer   7.26
ushcn_12mo      grud        -4.69
                mtand        2.14
                warpformer  -0.07
ushcn_1mo       grud         9.45
                mtand        0.66
                warpformer   4.01
AVERAGE                     18.95

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.33932,,,,
opera,0.33052,+18.95%,51/60,85%,10/12


## Irregular methods — MAE

In [8]:
show(fc, "mae", 5, grp="irregular_backbone")


========== MAE — Clean (no shift) ==========
Values (mean ± std over seeds)


method                                vanilla              opera
setting         backbone                                        
activity_1000ms grud        0.03536 ± 0.00033  0.03643 ± 0.00038
                mtand       0.03642 ± 0.00080  0.03674 ± 0.00030
                warpformer  0.07722 ± 0.00483  0.07095 ± 0.00369
physionet_24h   grud        0.04077 ± 0.00068  0.04071 ± 0.00072
                mtand       0.04533 ± 0.00117  0.04571 ± 0.00076
                warpformer  0.05174 ± 0.00054  0.05116 ± 0.00034
ushcn_12mo      grud        0.39791 ± 0.00988  0.41457 ± 0.00967
                mtand       0.39997 ± 0.01873  0.40385 ± 0.02226
                warpformer  0.43444 ± 0.02302  0.43928 ± 0.01906
ushcn_1mo       grud        0.31459 ± 0.00208  0.31334 ± 0.00233
                mtand       0.43193 ± 0.00961  0.43441 ± 0.00392
                warpformer  0.34651 ± 0.00645  0.34061 ± 0.00487
AVERAGE                               0.21768            0.21898

% improvement vs baseline (positive = better)


method                      opera
setting         backbone         
activity_1000ms grud        -3.03
                mtand       -0.66
                warpformer   9.43
physionet_24h   grud         0.17
                mtand       -0.11
                warpformer   0.76
ushcn_12mo      grud        -4.12
                mtand       -1.05
                warpformer  -1.03
ushcn_1mo       grud         0.40
                mtand       -1.52
                warpformer   1.84
AVERAGE                      0.09

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.21768,,,,
opera,0.21898,+0.09%,28/60,47%,5/12


... (3 outputs left out)

% improvement vs baseline (positive = better)


method                      opera
setting         backbone         
activity_1000ms grud        48.50
                mtand       40.64
                warpformer  18.36
physionet_24h   grud        11.57
                mtand        8.97
                warpformer   3.55
ushcn_12mo      grud        -2.42
                mtand        1.02
                warpformer  -0.23
ushcn_1mo       grud         8.27
                mtand       -0.39
                warpformer   3.43
AVERAGE                     11.77

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,0.24205,,,,
opera,0.22999,+11.77%,47/60,78%,10/12


## Irregular methods — AUROC

In [9]:
show(cl, "auroc", 2, grp="irregular_backbone")


========== AUROC — Clean (no shift) ==========
Values (mean ± std over seeds)


method                   vanilla         opera
setting backbone                              
P12     grud        83.72 ± 1.33  84.24 ± 1.41
        mtand       86.13 ± 1.08  86.88 ± 0.72
        warpformer  76.62 ± 1.28  78.36 ± 0.81
P19     grud        89.02 ± 0.59  89.64 ± 0.65
        mtand       79.82 ± 1.53  80.09 ± 2.19
        warpformer  87.91 ± 0.84  87.03 ± 1.02
AVERAGE                    83.87         84.37

% improvement vs baseline (positive = better)


method              opera
setting backbone         
P12     grud         0.79
        mtand        1.04
        warpformer   1.87
P19     grud         0.79
        mtand       -1.08
        warpformer  -0.76
AVERAGE              0.44

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,83.87,,,,
opera,84.37,+0.44%,19/30,63%,5/6


... (3 outputs left out)

% improvement vs baseline (positive = better)


method              opera
setting backbone         
P12     grud         1.91
        mtand        1.30
        warpformer   3.14
P19     grud         2.71
        mtand        2.57
        warpformer   0.87
AVERAGE              2.08

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,79.29,,,,
opera,81.45,+2.08%,30/30,100%,6/6


## Irregular methods — F1 (macro)

In [10]:
show(cl, "f1", 2, grp="irregular_backbone")


========== F1 — Clean (no shift) ==========


Values (mean ± std over seeds)


method                   vanilla         opera
setting backbone                              
P12     grud        67.76 ± 0.48  69.42 ± 1.57
        mtand       69.20 ± 0.62  70.39 ± 0.96
        warpformer  60.95 ± 0.81  62.17 ± 0.54
P19     grud        70.56 ± 1.89  70.43 ± 1.13
        mtand       60.62 ± 0.49  59.83 ± 0.97
        warpformer  66.81 ± 1.83  69.71 ± 2.04
PAM     grud        89.71 ± 1.59  90.83 ± 0.51
        mtand       90.55 ± 0.78  91.18 ± 0.54
        warpformer  73.60 ± 3.93  77.98 ± 1.56
AVERAGE                    72.20         73.55

% improvement vs baseline (positive = better)


method              opera
setting backbone         
P12     grud         2.22
        mtand        1.27
        warpformer   1.87
P19     grud        -1.10
        mtand       -0.91
        warpformer   4.12
PAM     grud         1.31
        mtand        0.76
        warpformer   6.13
AVERAGE              1.74

Summary per method

... (3 outputs left out)

% improvement vs baseline (positive = better)


method              opera
setting backbone         
P12     grud         5.66
        mtand        2.23
        warpformer   4.42
P19     grud        -5.59
        mtand        2.57
        warpformer   9.10
PAM     grud        10.61
        mtand        7.21
        warpformer   9.71
AVERAGE              5.10

Summary per method


,avg value,avg % improvement,seed wins,seed win %,cell wins
vanilla,63.55,,,,
opera,67.05,+5.10%,40/45,89%,8/9


# All backbones side by side

Our backbones (iTransformer, PatchTST) next to the irregular methods (GRU-D, mTAND, Warpformer), baseline vs
OPERA on the same 7 settings. Then one summary line per backbone: average % improvement and seed wins.

In [11]:
BB = ["itransformer", "patchtst", "grud", "mtand", "warpformer"]

def side_by_side(df, metric, digits):
    sign = -1 if metric in LOWER_BETTER else 1
    for cond, name in (("clean", "Clean"), ("mean_4shifts", "Shifted (mean of 4)")):
        d = df[df.group.isin(["main", "irregular_backbone"]) & (df.condition == cond)
               & df.method.isin(["vanilla", "opera"]) & df.backbone.isin(BB)].dropna(subset=[f"{metric}_mean"])
        if d.empty: continue
        t = d.pivot_table(index=["setting", "backbone"], columns="method", values=f"{metric}_mean")
        t["% improvement"] = d[d.method == "opera"].set_index(["setting", "backbone"])[f"{metric}_vs_baseline_pct"]
        t = t.reindex(sorted(t.index, key=lambda k: (k[0], BB.index(k[1]))))[["vanilla", "opera", "% improvement"]]
        print(f"\n========== {metric.upper()} — {name} =========="); display(t.round({"vanilla": digits, "opera": digits, "% improvement": 2}))
        sv = seed_values(metric, "clean" if cond == "clean" else "shift", "main")
        sv = pd.concat([sv, seed_values(metric, "clean" if cond == "clean" else "shift", "irregular_backbone")])
        rows = {}
        for b in BB:
            p = sv.xs(b, level="backbone")[["opera", "vanilla"]].dropna() if b in sv.index.get_level_values("backbone") else pd.DataFrame()
            if p.empty: continue
            w = int((sign * (p["opera"] - p["vanilla"]) > 0).sum())
            rows[b] = {"family": "ours" if b in ("itransformer", "patchtst") else "irregular method",
                       "avg % improvement": f"{t.xs(b, level='backbone')['% improvement'].mean():+.2f}%",
                       "seed wins": f"{w}/{len(p)}", "seed win %": f"{100 * w / len(p):.0f}%"}
        print("Per backbone"); display(pd.DataFrame(rows).T)

## Side by side — MSE

In [12]:
side_by_side(fc, "mse", 5)


========== MSE — Clean ==========


method                        vanilla    opera  % improvement
setting         backbone                                     
activity_1000ms itransformer  0.00266  0.00287          -7.31
                patchtst      0.00287  0.00307          -7.00
                grud          0.00292  0.00301          -3.02
                mtand         0.00307  0.00310          -0.91
                warpformer    0.01037  0.00873          17.13
physionet_24h   itransformer  0.00547  0.00550          -2.46
                patchtst      0.01157  0.01204          -2.92
                grud          0.00576  0.00567           1.87
                mtand         0.00653  0.00653           0.76
                warpformer    0.00844  0.00828           1.80
ushcn_12mo      itransformer  0.52247  0.51597           1.24
                patchtst      0.51645  0.52334          -1.43
                grud          0.62322  0.66091          -5.60
                mtand         0.62400  0.63149          -0.97
                warpformer    0.68340  0.69312          -0.83
ushcn_1mo       itransformer  0.52948  0.51638           1.86
                patchtst      0.51211  0.51580          -0.71
                grud          0.52140  0.51782           0.66
                mtand         0.70096  0.69998          -0.00
                warpformer    0.55597  0.55166           1.48

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,-1.67%,11/20,55%
patchtst,ours,-3.01%,1/20,5%
grud,irregular method,-1.52%,8/20,40%
mtand,irregular method,-0.28%,6/20,30%
warpformer,irregular method,+4.89%,15/20,75%



========== MSE — Shifted (mean of 4) ==========


method                        vanilla    opera  % improvement
setting         backbone                                     
activity_1000ms itransformer  0.06671  0.00437          93.46
                patchtst      0.09264  0.01168          91.84
                grud          0.01945  0.00443          76.19
                mtand         0.01239  0.00422          66.43
                warpformer    0.01342  0.00905          34.10
physionet_24h   itransformer  0.01888  0.00670          63.81
                patchtst      0.05413  0.03714          30.43
                grud          0.00850  0.00706          15.91
                mtand         0.00886  0.00742          16.02
                warpformer    0.00939  0.00878           7.26
ushcn_12mo      itransformer  0.55255  0.53567           3.55
                patchtst      0.61245  0.59361           3.57
                grud          0.65255  0.68006          -4.69
                mtand         0.65776  0.64570           2.14
                warpformer    0.70592  0.70995          -0.07
ushcn_1mo       itransformer  0.56403  0.53834           3.63
                patchtst      0.61290  0.58845           4.06
                grud          0.65212  0.59014           9.45
                mtand         0.72160  0.71481           0.66
                warpformer    0.60993  0.58461           4.01

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,+41.11%,20/20,100%
patchtst,ours,+32.48%,20/20,100%
grud,irregular method,+24.21%,15/20,75%
mtand,irregular method,+21.31%,19/20,95%
warpformer,irregular method,+11.33%,17/20,85%


## Side by side — MAE

In [13]:
side_by_side(fc, "mae", 5)


========== MAE — Clean ==========

method                        vanilla    opera  % improvement
setting         backbone                                     
activity_1000ms itransformer  0.03202  0.03341          -4.73
                patchtst      0.03260  0.03385          -3.57
                grud          0.03536  0.03643          -3.03
                mtand         0.03642  0.03674          -0.66
                warpformer    0.07722  0.07095           9.43
physionet_24h   itransformer  0.03805  0.03799          -0.00
                patchtst      0.05451  0.05752          -3.05
                grud          0.04077  0.04071           0.17
                mtand         0.04533  0.04571          -0.11
                warpformer    0.05174  0.05116           0.76
ushcn_12mo      itransformer  0.32375  0.31759           0.62
                patchtst      0.30461  0.30674          -0.08
                grud          0.39791  0.41456          -4.12
                mtand         0.39997  0.40385          -1.05
                warpformer    0.43444  0.43928          -1.03
ushcn_1mo       itransformer  0.32423  0.31420           1.51
                patchtst      0.30860  0.30621           1.23
                grud          0.31459  0.31334           0.40
                mtand         0.43193  0.43441          -1.52
                warpformer    0.34651  0.34061           1.84

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,-0.65%,9/20,45%
patchtst,ours,-1.37%,8/20,40%
grud,irregular method,-1.65%,7/20,35%
mtand,irregular method,-0.84%,6/20,30%
warpformer,irregular method,+2.75%,15/20,75%



========== MAE — Shifted (mean of 4) ==========

method                        vanilla    opera  % improvement
setting         backbone                                     
activity_1000ms itransformer  0.22170  0.04313          80.52
                patchtst      0.26714  0.06427          79.02
                grud          0.09007  0.04559          48.50
                mtand         0.07320  0.04358          40.64
                warpformer    0.08663  0.07180          18.36
physionet_24h   itransformer  0.07507  0.04283          42.94
                patchtst      0.14076  0.10252          27.46
                grud          0.05178  0.04542          11.57
                mtand         0.05268  0.04859           8.97
                warpformer    0.05471  0.05279           3.55
ushcn_12mo      itransformer  0.33428  0.32185           1.75
                patchtst      0.37674  0.35971           4.79
                grud          0.41670  0.42579          -2.42
                mtand         0.41569  0.40997           1.02
                warpformer    0.45329  0.45672          -0.23
ushcn_1mo       itransformer  0.34869  0.32707           6.39
                patchtst      0.37699  0.35355           6.80
                grud          0.38676  0.35499           8.27
                mtand         0.44464  0.44156          -0.39
                warpformer    0.37849  0.36309           3.43

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,+32.90%,20/20,100%
patchtst,ours,+29.52%,18/20,90%
grud,irregular method,+16.48%,15/20,75%
mtand,irregular method,+12.56%,16/20,80%
warpformer,irregular method,+6.28%,16/20,80%


## Side by side — AUROC

In [14]:
side_by_side(cl, "auroc", 2)


========== AUROC — Clean ==========


method                vanilla  opera  % improvement
setting backbone                                   
P12     itransformer    84.11  84.72           0.07
        patchtst        85.03  85.75           0.95
        grud            83.72  84.24           0.79
        mtand           86.13  86.88           1.04
        warpformer      76.62  78.36           1.87
P19     itransformer    87.30  89.02           1.31
        patchtst        82.48  83.57           0.37
        grud            89.02  89.64           0.79
        mtand           79.82  80.09          -1.08
        warpformer      87.91  87.03          -0.76

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,+0.69%,8/10,80%
patchtst,ours,+0.66%,9/10,90%
grud,irregular method,+0.79%,8/10,80%
mtand,irregular method,-0.02%,6/10,60%
warpformer,irregular method,+0.56%,5/10,50%



========== AUROC — Shifted (mean of 4) ==========

method                vanilla  opera  % improvement
setting backbone                                   
P12     itransformer    80.62  82.32           0.65
        patchtst        81.94  83.31           1.70
        grud            80.49  82.33           1.91
        mtand           82.93  84.17           1.30
        warpformer      74.60  77.15           3.14
P19     itransformer    84.21  86.93           3.52
        patchtst        80.31  82.48           2.26
        grud            81.74  84.46           2.71
        mtand           74.38  77.29           2.57
        warpformer      81.60  83.29           0.87

Per backbone

,family,avg % improvement,seed wins,seed win %
itransformer,ours,+2.09%,10/10,100%
patchtst,ours,+1.98%,10/10,100%
grud,irregular method,+2.31%,10/10,100%
mtand,irregular method,+1.93%,10/10,100%
warpformer,irregular method,+2.01%,10/10,100%


## Side by side — F1 (macro)

In [15]:
side_by_side(cl, "f1", 2)


========== F1 — Clean ==========

method                vanilla  opera  % improvement
setting backbone                                   
P12     itransformer    68.24  67.68          -0.62
        patchtst        68.71  69.37           1.03
        grud            67.76  69.42           2.22
        mtand           69.20  70.39           1.27
        warpformer      60.95  62.17           1.87
P19     itransformer    66.93  65.38          -1.64
        patchtst        65.82  62.45          -7.84
        grud            70.56  70.43          -1.10
        mtand           60.62  59.83          -0.91
        warpformer      66.81  69.71           4.12
PAM     itransformer    89.75  89.83           0.04
        patchtst        88.40  89.00          -0.41
        grud            89.71  90.83           1.31
        mtand           90.55  91.18           0.76
        warpformer      73.60  77.98           6.13

Per backbone


,family,avg % improvement,seed wins,seed win %
itransformer,ours,-0.74%,5/13,38%
patchtst,ours,-2.40%,6/13,46%
grud,irregular method,+0.81%,11/15,73%
mtand,irregular method,+0.37%,10/15,67%
warpformer,irregular method,+4.04%,15/15,100%



========== F1 — Shifted (mean of 4) ==========


method                vanilla  opera  % improvement
setting backbone                                   
P12     itransformer    65.32  66.35           1.70
        patchtst        66.25  67.33           1.53
        grud            63.60  67.10           5.66
        mtand           66.71  68.52           2.23
        warpformer      59.11  61.67           4.42
P19     itransformer    69.64  70.16           1.14
        patchtst        63.07  61.36          -4.95
        grud            64.46  60.99          -5.59
        mtand           52.48  54.03           2.57
        warpformer      54.25  59.57           9.10
PAM     itransformer    68.95  79.55          12.76
        patchtst        73.56  77.02           5.18
        grud            71.89  80.43          10.61
        mtand           77.22  82.87           7.21
        warpformer      62.22  68.29           9.71

Per backbone


,family,avg % improvement,seed wins,seed win %
itransformer,ours,+5.20%,11/13,85%
patchtst,ours,+0.59%,10/13,77%
grud,irregular method,+3.56%,10/15,67%
mtand,irregular method,+4.00%,15/15,100%
warpformer,irregular method,+7.74%,15/15,100%
